# Import Library & Load Dataset

In [1]:
import os
import pandas as pd

BASE_DIR = '../'
CSV_FILES = [
    os.path.join(BASE_DIR, "dataset_raw", "goemotions_1.csv"),
    os.path.join(BASE_DIR, "dataset_raw", "goemotions_2.csv"),
    os.path.join(BASE_DIR, "dataset_raw", "goemotions_3.csv"),
]

dfs = [pd.read_csv(file) for file in CSV_FILES]
df_all = pd.concat(dfs, ignore_index=True)

print(f"Total baris setelah digabung: {len(df_all):,} baris")

Total baris setelah digabung: 211,225 baris


# Filtering & Label Extraction

In [ ]:
# 6 Emosi Inti
CORE_EMOTIONS = ['joy', 'love', 'surprise', 'anger', 'fear', 'sadness']
COLS_TO_KEEP = ['text'] + CORE_EMOTIONS
df_all = df_all[COLS_TO_KEEP].copy()

# Urutan prioritas ketika satu baris multi-label
PRIORITY_ORDER = ['joy', 'love', 'surprise', 'anger', 'fear', 'sadness']

def get_pure_emotion(row):
    for emotion in PRIORITY_ORDER:
        if row[emotion] == 1:
            return emotion.capitalize()
    return None # Return None jika tidak ada satupun dari 6 emosi ini yang bernilai 1

df_all['emotion'] = df_all.apply(get_pure_emotion, axis=1)

# Data Cleaning & Distribution Check

In [ ]:
# Hapus baris kosong (emosi netral & 21 emosi lain yang jadi None)
before_drop = len(df_all)
df_final = df_all.dropna(subset=['emotion'])[['text', 'emotion']].copy()
df_final.reset_index(drop=True, inplace=True)
after_drop = len(df_final)

print(f"Baris yang dibuang (Bukan 6 emosi inti murni) : {before_drop - after_drop:,} baris")
print(f"Sisa baris kualitas tinggi untuk training      : {after_drop:,} baris")

# Ringkasan distribusi baru
print("\nDISTRIBUSI 6 KELAS EMOSI INTI MURNI:")
dist = df_final["emotion"].value_counts()
total = len(df_final)

for emotion, count in dist.items():
    bar = "█" * int(count / total * 40)
    print(f"{emotion:<10} : {count:>6,} baris ({count/total*100:5.1f}%)  {bar}")

Baris yang dibuang (Bukan 6 emosi inti murni) : 172,967 baris
Sisa baris kualitas tinggi untuk training      : 38,258 baris

DISTRIBUSI 6 KELAS EMOSI INTI MURNI:
Joy        :  7,983 baris ( 20.9%)  ████████
Anger      :  7,926 baris ( 20.7%)  ████████
Love       :  7,756 baris ( 20.3%)  ████████
Sadness    :  6,252 baris ( 16.3%)  ██████
Surprise   :  5,322 baris ( 13.9%)  █████
Fear       :  3,019 baris (  7.9%)  ███


In [4]:
print("Preview 5 baris pertama yang baru ditambahkan:")
print(df_final.head())
print("Preview 5 baris terakhir yang baru ditambahkan:")
print(df_final.tail())

Preview 5 baris pertama yang baru ditambahkan:
                                                text  emotion
0                                    That game hurt.  Sadness
1                                 Man I love reddit.     Love
2  So happy for [NAME]. So sad he's not here. Ima...      Joy
3  I just came home, what the fuck is this lineup...     Love
4  By far the coolest thing I've seen on this thr...      Joy
Preview 5 baris terakhir yang baru ditambahkan:
                                                    text emotion
38253  I just called the Capitol Police. They are not...   Anger
38254    What a great photo and you two look so happy. 😍     Joy
38255  Well, I'm glad you're out of all that now. How...     Joy
38256                             Everyone likes [NAME].    Love
38257  The FDA has plenty to criticize. But like here...   Anger


# Export to CSV

In [5]:
OUTPUT_DIR = "../data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(OUTPUT_DIR, "goemotions_kasar_pure.csv")

df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(f"Dataset tersimpan di: {OUTPUT_PATH}")

Dataset tersimpan di: ../data\goemotions_kasar_pure.csv
